In [0]:
# Filename: src/notebooks/bronze_ingestion.py
from pyspark.sql.functions import current_timestamp, input_file_name, col

# Define paths (Using the Volume established)
source_path = "/Volumes/workspace/default/ecommerce_analytics_dev/bronze_layer/raw_files/"
target_table = "workspace.bronze_layer.events_raw"

# Read raw CSV with schema inference [cite: 126]
raw_df = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(source_path))

# Add mandatory audit columns [cite: 56]
bronze_df = (raw_df
    .withColumn("ingestion_timestamp", current_timestamp())
    # Use _metadata.file_path instead of input_file_name()
    .withColumn("source_file", col("_metadata.file_path"))
)
# Write to Delta Bronze Layer [cite: 119]
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze_layer")
bronze_df.write.format("delta").mode("append").saveAsTable(target_table)


In [0]:
from pyspark.sql import functions as f

# Extract metadata from Bronze table to create a high-level audit summary
audit_df = spark.read.table("workspace.default.events_raw") \
    .select("source_file", "ingestion_timestamp") \
    .distinct() \
    .withColumn("process_name", f.lit("Bronze_Ingestion_Job")) \
    .withColumn("status", f.lit("SUCCESS"))

audit_df.write.format("delta").mode("append").saveAsTable("workspace.bronze_layer.ingestion_audit_logs")